In [ ]:
import os, shutil, subprocess
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN").strip()
assert tok.startswith(("github_pat_", "ghp_")), "GH_TOKEN secret must be ONLY the token, no url/spaces"

# HF_TOKEN: authenticated HF Hub downloads (higher rate limits, faster model pulls)
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN").strip()
except Exception:
    print("no HF_TOKEN secret — HF downloads unauthenticated (slower, rate-limited)")

url = f"https://{tok}@github.com/erkenovaj/building-connections.git"
print("clone url:", url.replace(tok, "***"))

os.chdir("/kaggle/working")          # leave bc before deleting it
shutil.rmtree("/kaggle/working/bc", ignore_errors=True)
r = subprocess.run(
    ["git", "clone", "--branch", "feat/llm-agent-loop", "--single-branch", url, "/kaggle/working/bc"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise SystemExit("git clone failed:\n" + (r.stderr or "").replace(tok, "***"))
print("clone OK")
%cd /kaggle/working/bc
!git log --oneline -3

In [ ]:
!pip install -q -r requirements-train.txt
!mkdir -p data outputs

In [ ]:
import torch
assert torch.__version__.startswith("2.10."), f"wrong torch {torch.__version__} — rerun the requirements cell"
assert torch.cuda.is_available(), "no CUDA — check accelerator setting / torch build"
print(torch.__version__, "|", torch.cuda.get_device_name(0))

## Stage 1: 2-category curriculum

**Step 1 — probe gate.** Proceed only if `win_at_k["1"] >= 0.05` (GRPO/STaR need successes to amplify). If 0 — investigate prompt/budget before training. Probe pool (96 boards) must be >= GRPO's `--num-boards` (64) so difficulty ordering has a slice to select from; `--out-boards` feeds the GRPO `--seed-order`.

In [ ]:
!python train/probe_wink.py --model Qwen/Qwen3-1.7B --num-categories 2 \
  --num-boards 96 --num-episodes 4 --max-new-tokens 2048 \
  --out-boards data/boards_2cat.jsonl

**Step 2 — STaR SFT.** Sample base-model episodes, keep only wins, fit LoRA on them (loss on assistant tokens only).

In [ ]:
# 1200 episodes (~6x the old 200): 17 kept trajectories are too few to
# tell SFT from DFT apart; at the observed ~8.5% keep rate expect ~100.
!python train/star_sft.py sample --model Qwen/Qwen3-1.7B --num-categories 2 \
  --seed-start 150 --num-boards 150 --episodes-per-board 8 \
  --max-new-tokens 2048 --out /kaggle/working/star_2cat_kaggle.jsonl

In [ ]:
# --epochs 3: 17 examples gave only 4 optimizer steps; with ~100 kept expect ~21
!python train/star_sft.py train --model Qwen/Qwen3-1.7B --epochs 3 \
  --data /kaggle/working/star_2cat_kaggle.jsonl --output outputs/star-sft-2cat

In [ ]:
# re-probe with the STaR adapter: expect win@1 up vs Step 1
# seeds 2000000+ are held out from the GRPO pool (1000000-1000095) — no eval leakage
!python train/probe_wink.py --model Qwen/Qwen3-1.7B --adapter outputs/star-sft-2cat \
  --num-categories 2 --num-boards 32 --num-episodes 4 \
  --seed-start 2000000 --max-new-tokens 2048

**Step 2b — DFT branch (mentor's comparison).** Train a second LoRA on the SAME won-trajectory JSONL, but with DFT loss `-sg(p)*log p` (arXiv:2508.05629) instead of NLL. Endpoint: DFT probe below vs the post-GRPO gate in Step 3 — same 32 boards, 4 episodes. If DFT train loss stays flat, bump `--lr 2e-5` (the sg(p) weight shrinks gradients).

In [ ]:
# DFT branch: same JSONL and epochs as the SFT cell above, only the loss differs.
# If train loss barely moves, retry with --lr 2e-5 (sg(p) <= 1 shrinks gradients).
!python train/star_sft.py train --model Qwen/Qwen3-1.7B --dft --epochs 3 \
  --data /kaggle/working/star_2cat_kaggle.jsonl --output outputs/star-dft-2cat

In [ ]:
# DFT probe: same 32 held-out boards as the STaR probe above, numbers directly comparable.
# Mentor's comparison = this vs the post-GRPO gate in Step 3.
!python train/probe_wink.py --model Qwen/Qwen3-1.7B --adapter outputs/star-dft-2cat \
  --num-categories 2 --num-boards 32 --num-episodes 4 \
  --seed-start 2000000 --max-new-tokens 2048

**Step 3 — agentic GRPO from the STaR adapter.** `--seed-order` selects the 64 easiest probed boards as the training pool. Watch: reward EMA rising, no persistent zero-grad lines, truncation low.

In [ ]:
# --max-new-tokens 2048 matches the sample/probe budget; if the P100 OOMs, lower --batch-size
!TRL_EXPERIMENTAL_SILENCE=1 python train/grpo_agentic.py --model Qwen/Qwen3-1.7B \
  --num-categories 2 --init-lora outputs/star-sft-2cat \
  --seed-order data/boards_2cat.jsonl --max-steps 50 --max-new-tokens 2048 \
  --output outputs/grpo-agentic-2cat

In [ ]:
# post-GRPO gate: advance to 3-cat when win_at_k["1"] >= 0.6 (same held-out seeds as SFT/DFT probes)
!python train/probe_wink.py --model Qwen/Qwen3-1.7B --adapter outputs/grpo-agentic-2cat \
  --num-categories 2 --num-boards 32 --num-episodes 4 \
  --seed-start 2000000 --max-new-tokens 2048

## Stage 2: 3 categories (run only after the 2-cat gate passes)

Each stage starts from the previous adapter via `--init-lora`, re-probes with a fresh `--out-boards`, trains with the matching `--seed-order`. If win@k collapsed at the new size, insert a STaR round (as in Stage 1) before GRPO.

In [ ]:
!python train/probe_wink.py --model Qwen/Qwen3-1.7B --adapter outputs/grpo-agentic-2cat \
  --num-categories 3 --num-boards 96 --num-episodes 4 --max-new-tokens 2048 \
  --out-boards data/boards_3cat.jsonl

In [ ]:
!TRL_EXPERIMENTAL_SILENCE=1 python train/grpo_agentic.py --model Qwen/Qwen3-1.7B \
  --num-categories 3 --init-lora outputs/grpo-agentic-2cat \
  --seed-order data/boards_3cat.jsonl --max-steps 50 --max-new-tokens 2048 \
  --output outputs/grpo-agentic-3cat

In [ ]:
# 3-cat gate: advance to 4-cat when win_at_k["1"] >= 0.6 (held-out eval seeds)
!python train/probe_wink.py --model Qwen/Qwen3-1.7B --adapter outputs/grpo-agentic-3cat \
  --num-categories 3 --num-boards 32 --num-episodes 4 \
  --seed-start 2000000 --max-new-tokens 2048

## Stage 3: full 4-category game (run only after the 3-cat gate passes)

In [ ]:
!python train/probe_wink.py --model Qwen/Qwen3-1.7B --adapter outputs/grpo-agentic-3cat \
  --num-categories 4 --num-boards 96 --num-episodes 4 --max-new-tokens 2048 \
  --out-boards data/boards_4cat.jsonl

In [ ]:
!TRL_EXPERIMENTAL_SILENCE=1 python train/grpo_agentic.py --model Qwen/Qwen3-1.7B \
  --num-categories 4 --init-lora outputs/grpo-agentic-3cat \
  --seed-order data/boards_4cat.jsonl --max-steps 50 --max-new-tokens 2048 \
  --output outputs/grpo-agentic-4cat

In [ ]:
# final eval (held-out seeds)
!python train/probe_wink.py --model Qwen/Qwen3-1.7B --adapter outputs/grpo-agentic-4cat \
  --num-categories 4 --num-boards 32 --num-episodes 4 \
  --seed-start 2000000 --max-new-tokens 2048